# Previsão de jogos da NBA com Redes Neurais Líquidas (CfC)

Este notebook roda o projeto completo no Google Colab: features → treino (CfC, LTC, NCP, LSTM e baselines) → figuras.

Os dados brutos (`data/raw/`) já estão no repositório. Eles foram baixados localmente com `python -m src.data.collect`, porque o stats.nba.com bloqueia IPs do Colab.

**Para rodar:** *Ambiente de execução → Executar tudo*. GPU é opcional: o modelo é pequeno e roda bem em CPU.

In [ ]:
# 1. Clonar o repositório e instalar as dependências
REPO_URL = "https://github.com/Jlucas66/Preview-of-NBA-games.git"

import os, subprocess, sys
if not os.path.exists("Preview-of-NBA-games"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "Preview-of-NBA-games"], check=True)
os.chdir("Preview-of-NBA-games/nba-lnn")   # o projeto fica na subpasta nba-lnn/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ncps", "nba_api"], check=True)

def run(*args):
    """Roda `python -m <args>` mostrando a saída; para o notebook se der erro."""
    r = subprocess.run([sys.executable, "-m", *args], capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError(f"falhou: python -m {' '.join(args)}")

In [ ]:
# (opcional) Sem os dados reais no repositório? Gera dados sintéticos só para testar o pipeline.
if not os.path.exists("data/raw") or not os.listdir("data/raw"):
    print("data/raw vazio -> gerando dados SINTÉTICOS (resultados não valem para a NBA real)")
    run("src.data.collect", "--synthetic")

## 2. Features: Elo, histórico, impacto dos jogadores, desfalques, lesões

In [ ]:
run("src.features.build")

In [ ]:
import pandas as pd
games = pd.read_csv("data/processed/games.csv")
games.head()

## 3. Treino e comparação dos modelos (5 sementes por rede)

In [ ]:
run("src.train", "--model", "all", "--seeds", "5", "--quiet")

In [ ]:
res = pd.read_csv("artifacts/results.csv")
res[["modelo", "accuracy", "accuracy_std", "log_loss", "log_loss_std", "brier", "auc", "params"]].round(4)

## 4. Figuras para o artigo

In [ ]:
run("src.analysis")

In [ ]:
from IPython.display import Image, display
for f in ["comparacao_modelos", "curvas_treino", "calibracao", "importancia", "efeito_desfalque"]:
    print(f)
    display(Image(f"artifacts/figures/{f}.png"))